In [1]:
from pathlib import Path
import pandas as pd
import json
import os

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
top_path = Path(os.path.dirname(os.getcwd()))
data_path = top_path / "data"
insights_path = top_path / "reports" / "insights"

notebooks_path = top_path / "notebooks"
team_data_path = data_path / "processed" / "team_data.parquet"
predictions_path = insights_path / "LightGBM_predictions.parquet"

In [3]:
predictions_df = pd.read_parquet(predictions_path).rename(columns={"result": "prediction"})
team_df = pd.read_parquet(team_data_path)

## Store the accuracy per league in a dictionary

In [4]:
league_data = team_df[["gameid", "side", "league", "result"]]
predictions_df = predictions_df.merge(league_data, on=["gameid", "side"], how="inner", validate="many_to_many")

In [5]:
# Explore accuracy by league
leagues = predictions_df["league"].unique()
accuracies = {}

for league in leagues:
    league_df = predictions_df[predictions_df["league"] == league]
    accuracy = league_df["prediction"] == league_df["result"]
    accuracies[league] = {"count": len(league_df), "accuracy": accuracy.mean()}

accuracies = {k: v for k, v in sorted(accuracies.items(), key=lambda item: item[1]["accuracy"], reverse=True)}

with open(insights_path / "LightGBM_league_accuracies.json", "w") as f:
    json.dump(accuracies, f)

# Specific League Predictions Analysis

In [6]:
# Specific League Analysis
analysis_league = "LEC"
games_data =  team_df[["date", "gameid", "teamname", "opponentteam", "side"]]

lec_df = predictions_df[predictions_df["league"] == analysis_league][["gameid", "side", "league", "prediction", "result"]]
lec_df = games_data.merge(lec_df, on=["gameid", "side"])

lec_df["correct"] = lec_df["prediction"] == lec_df["result"]
lec_df.sort_values(by="date", inplace=True)

lec_df

,date,gameid,teamname,opponentteam,side,league,prediction,result,correct
0,2022-01-14 20:10:24,ESPORTSTMNT04_2090358,Fnatic,Team BDS,Blue,LEC,1,1,True
1,2022-01-14 20:10:24,ESPORTSTMNT04_2090358,Team BDS,Fnatic,Red,LEC,0,0,True
2,2022-01-16 15:02:39,ESPORTSTMNT01_2702406,Team BDS,Misfits Gaming,Blue,LEC,0,0,True
3,2022-01-16 15:02:39,ESPORTSTMNT01_2702406,Misfits Gaming,Team BDS,Red,LEC,1,1,True
4,2022-01-16 18:53:12,ESPORTSTMNT01_2692441,MAD Lions,G2 Esports,Blue,LEC,0,1,False
...,...,...,...,...,...,...,...,...,...
305,2024-06-15 15:07:41,LOLTMNT04_52448,SK Gaming,Team BDS,Red,LEC,0,1,False
306,2024-06-15 16:02:24,LOLTMNT04_52452,Fnatic,Rogue,Blue,LEC,1,1,True
307,2024-06-15 16:02:24,LOLTMNT04_52452,Rogue,Fnatic,Red,LEC,0,0,True
308,2024-06-22 17:37:36,LOLTMNT05_52466,G2 Esports,Karmine Corp,Blue,LEC,1,1,True
